# Лабораторная работа № 1. Массивы NumPy

**Цель работы:** освоить создание, индексацию и базовые операции с массивами NumPy на данных о продажах автомобилей.

**Данные:** статистика продаж автомобилей на вторичном рынке Молдавии (`cars.csv`). Этот же набор данных используется в лабораторной работе № 2 (pandas) и в последующих работах курса, посвящённых предсказанию цены автомобиля.

**Порядок выполнения:**
1. Ячейки выполняются последовательно сверху вниз.
2. В ячейках упражнений многоточие `...` заменяется кодом.
3. Ячейка с проверками `assert` выполняется без ошибок, если упражнение выполнено верно.
4. На вопросы, отмеченные **Вопрос**, дается письменный ответ в ячейке ниже вопроса.

## 0. Подготовка

Библиотека NumPy импортируется под принятым сокращением `np`.

In [1]:
import sys
import time

import numpy as np

np.set_printoptions(suppress=True)   # вывод чисел без экспоненциальной записи
print('Версия NumPy:', np.__version__)

Версия NumPy: 2.4.4


Набор данных загружается готовой ячейкой. Загрузка таблиц подробно рассматривается в лабораторной работе № 2.

Первые строки файла `cars.csv`:

| Make | Model | Year | Style | Distance | Engine_capacity(cm3) | Fuel_type | AutoTransmission | Price(euro) |
|---|---|---|---|---|---|---|---|---|
| Toyota | Prius | 2011 | Hatchback | 195000.0 | 1800.0 | Hybrid | True | 7750.0 |
| Renault | Grand Scenic | 2014 | Universal | 135000.0 | 1500.0 | Diesel | False | 8550.0 |
| Volkswagen | Golf | 1998 | Hatchback | 1.0 | 1400.0 | Petrol | False | 2200.0 |

Термины, используемые в работе:
* **запись (объект)** — один автомобиль, одна строка таблицы;
* **признак** — одна характеристика автомобиля, один столбец таблицы (год выпуска, пробег, цена);
* **вектор признака** — все значения одного признака, упорядоченные по записям.

Из таблицы загружаются пять векторов признаков:
* `year` — год выпуска;
* `distance` — пробег, км;
* `engine` — объём двигателя, см³;
* `price` — цена, евро;
* `auto` — тип коробки передач (`True` — автоматическая, `False` — механическая).

In [2]:
URL = 'https://raw.githubusercontent.com/MVRonkin/BasicDataAnalysisCourse/refs/heads/main/Workshops/cars.csv'

# Числовые столбцы: Year, Distance, Engine_capacity(cm3), Price(euro)
year, distance, engine, price = np.genfromtxt(
    URL, delimiter=',', skip_header=1, usecols=(2, 4, 5, 8), unpack=True)

# Тип коробки передач: True — автоматическая
auto = np.genfromtxt(URL, delimiter=',', skip_header=1, usecols=7, dtype=str) == 'True'

In [3]:
print('Число записей:', len(price))
print('year    :', year[:5])
print('distance:', distance[:5])
print('engine  :', engine[:5])
print('price   :', price[:5])
print('auto    :', auto[:5])

Число записей: 41007
year    : [2011. 2014. 1998. 2012. 2006.]
distance: [195000. 135000.      1. 110000. 200000.]
engine  : [1800. 1500. 1400. 1500. 1600.]
price   : [7750. 8550. 2200. 6550. 4100.]
auto    : [ True False False False False]


## 1. Список и вектор. Назначение NumPy

**Список** (`list`) — встроенный тип Python. Список может содержать элементы разных типов: числа, строки, другие списки.

**Вектор NumPy** (тип `numpy.ndarray`) — упорядоченный набор чисел **одного типа**. Элементы вектора размещаются в памяти последовательно, одним блоком.

Далее список и вектор сравниваются на одних и тех же данных — ценах пяти первых автомобилей.

In [4]:
price_list = [7750.0, 8550.0, 2200.0, 6550.0, 4100.0]   # список
price_vec = np.array(price_list)                        # вектор, созданный из списка

print(type(price_list))
print(type(price_vec), price_vec.dtype)

<class 'list'>
<class 'numpy.ndarray'> float64


### 1.1. Семантика операций

Для списка и вектора одна и та же запись операции имеет разный смысл.

In [5]:
print('список * 2 :', price_list * 2)   # список повторяется два раза
print('вектор * 2 :', price_vec * 2)    # каждый элемент умножается на 2

список * 2 : [7750.0, 8550.0, 2200.0, 6550.0, 4100.0, 7750.0, 8550.0, 2200.0, 6550.0, 4100.0]
вектор * 2 : [15500. 17100.  4400. 13100.  8200.]


In [6]:
print('список + список:', price_list + price_list)  # списки склеиваются
print('вектор + вектор:', price_vec + price_vec)    # элементы складываются попарно

список + список: [7750.0, 8550.0, 2200.0, 6550.0, 4100.0, 7750.0, 8550.0, 2200.0, 6550.0, 4100.0]
вектор + вектор: [15500. 17100.  4400. 13100.  8200.]


In [7]:
try:
    price_list + 100
except TypeError as err:
    print('список + 100: ошибка ->', err)

print('вектор + 100:', price_vec + 100)  # число прибавляется к каждому элементу

список + 100: ошибка -> can only concatenate list (not "int") to list
вектор + 100: [7850. 8650. 2300. 6650. 4200.]


### 1.2. Перевод цен из евро в рубли

Для списка требуется цикл по элементам. Для вектора достаточно одной операции.

In [8]:
RATE = 100.0  # условный курс, руб./евро

# список: цикл
price_rub_list = []
for p in price_list:
    price_rub_list.append(p * RATE)
print(price_rub_list)

# вектор: одна операция
price_rub_vec = price_vec * RATE
print(price_rub_vec)

[775000.0, 855000.0, 220000.0, 655000.0, 410000.0]
[775000. 855000. 220000. 655000. 410000.]


### 1.3. Время вычислений

Сравнивается время перевода в рубли одного миллиона цен.

In [9]:
N = 1_000_000
big_vec = np.arange(N, dtype=float)   # вектор из N чисел 0.0, 1.0, ..., N - 1
big_list = big_vec.tolist()           # тот же набор чисел в виде списка

start = time.perf_counter()
res_list = []
for p in big_list:
    res_list.append(p * RATE)
time_list = time.perf_counter() - start

start = time.perf_counter()
res_vec = big_vec * RATE
time_vec = time.perf_counter() - start

print(f'список: {time_list:.4f} с')
print(f'вектор: {time_vec:.4f} с')
print(f'отношение: {time_list / time_vec:.0f}')

список: 0.0624 с
вектор: 0.0040 с
отношение: 16


### 1.4. Объём памяти

Список хранит ссылки на отдельные объекты-числа. Вектор хранит сами числа одним блоком.

In [10]:
mem_list = sys.getsizeof(big_list) + sum(sys.getsizeof(x) for x in big_list)
mem_vec = big_vec.nbytes

print(f'список: {mem_list / 2**20:.1f} МБ')
print(f'вектор: {mem_vec / 2**20:.1f} МБ')

список: 30.5 МБ


вектор: 7.6 МБ


### Упражнение 1

1. Переведите пробег всех автомобилей (`distance`) из километров в мили одной операцией. 1 миля = 1,609 км. Результат сохраните в переменной `distance_miles`.
2. **Вопрос.** Сформулируйте три отличия вектора NumPy от списка Python.

In [11]:
distance_miles = distance / 1.609

In [12]:
assert isinstance(distance_miles, np.ndarray)
assert len(distance_miles) == len(distance)
assert abs(distance_miles[0] - 195000 / 1.609) < 1e-6
print('Упражнение 1 выполнено')

Упражнение 1 выполнено


*Ответ на вопрос:*

## 2. Создание и индексация массивов

### 2.1. Создание массивов

| Функция | Результат |
|---|---|
| `np.array(список)` | массив из элементов списка |
| `np.arange(start, stop, step)` | числа от `start` до `stop` (не включая) с шагом `step` |
| `np.linspace(start, stop, num)` | `num` равноотстоящих чисел от `start` до `stop` (включая) |
| `np.zeros(n)`, `np.ones(n)` | `n` нулей, `n` единиц |

In [13]:
print(np.array([2011, 2014, 1998]))
print(np.arange(2010, 2021, 2))     # годы с 2010 по 2020 с шагом 2
print(np.linspace(0, 100_000, 5))   # 5 значений пробега от 0 до 100 000 км
print(np.zeros(4))
print(np.ones(4))

[2011 2014 1998]
[2010 2012 2014 2016 2018 2020]
[     0.  25000.  50000.  75000. 100000.]
[0. 0. 0. 0.]
[1. 1. 1. 1.]


Атрибуты массива:
* `shape` — форма (число элементов по каждой оси);
* `ndim` — число осей;
* `size` — общее число элементов;
* `dtype` — тип элементов.

In [14]:
print('shape:', price.shape)
print('ndim :', price.ndim)
print('size :', price.size)
print('dtype:', price.dtype, auto.dtype)

shape: (41007,)
ndim : 1
size : 41007
dtype: float64 bool


### 2.2. Индексация вектора

Нумерация элементов начинается с 0. Отрицательный индекс отсчитывается от конца. Срез `a[i:j]` включает элементы с номерами от `i` до `j - 1`.

In [15]:
print('первый элемент  :', price[0])
print('последний       :', price[-1])
print('элементы 0..4   :', price[:5])
print('элементы 10..14 :', price[10:15])
print('каждый 10 000-й :', price[::10_000])

первый элемент  : 7750.0
последний       : 4000.0
элементы 0..4   : [7750. 8550. 2200. 6550. 4100.]
элементы 10..14 : [11400. 11800.  1550.  3990.  5000.]
каждый 10 000-й : [7750. 9900. 8300. 2650. 2000.]


### 2.3. Матрица «записи × признаки»

Векторы признаков объединяются в двумерный массив (матрицу). Строка матрицы — запись (автомобиль), столбец — вектор признака.

Индексация матрицы: `X[строка, столбец]`. Двоеточие `:` означает «все элементы по этой оси».

In [16]:
X = np.column_stack((year, distance, engine, price))   # столбцы: год, пробег, объём, цена

print('shape:', X.shape)            # (число записей, число признаков)
print('запись 0      :', X[0, :])
print('признак «цена»:', X[:5, 3])  # первые 5 значений столбца 3
print('цена записи 0 :', X[0, 3])
print('первые 3 записи:')
print(X[:3, :])

shape: (41007, 4)
запись 0      : [  2011. 195000.   1800.   7750.]
признак «цена»: [7750. 8550. 2200. 6550. 4100.]
цена записи 0 : 7750.0
первые 3 записи:
[[  2011. 195000.   1800.   7750.]
 [  2014. 135000.   1500.   8550.]
 [  1998.      1.   1400.   2200.]]


### Упражнение 2.1

1. Создайте вектор `years_5` из годов 2000, 2005, ..., 2020 функцией `np.arange`.
2. Выберите из матрицы `X` запись с номером 100 в переменную `car_100`.
3. Выберите из матрицы `X` вектор признака «пробег» (столбец 1) в переменную `distance_from_X`.

In [17]:
years_5 = np.arange(2000, 2021, 5)
car_100 = X[100, :]
distance_from_X = X[:, 1]

In [18]:
assert list(years_5) == [2000, 2005, 2010, 2015, 2020]
assert car_100.shape == (4,) and car_100[3] == price[100]
assert distance_from_X.shape == distance.shape and (distance_from_X == distance).all()
print('Упражнение 2.1 выполнено')

Упражнение 2.1 выполнено


### 2.4. Отбор по условию

Сравнение вектора с числом дает **логическую маску** — вектор значений `True`/`False` той же длины. Индексация маской `a[mask]` оставляет элементы, для которых значение маски равно `True`.

Условия объединяются операциями `&` (и), `|` (или), `~` (не). Каждое условие заключается в скобки.

In [19]:
mask = price > 10_000
print('маска          :', mask[:5])
print('число записей  :', mask.sum())         # True считается как 1
print('цены > 10 000  :', price[mask][:5])

маска          : [False False False False False]
число записей  : 12716
цены > 10 000  : [17000. 11400. 11800. 15900. 27000.]


In [20]:
print('годы выпуска до 1970 г.        :', year[year < 1970][:10])
print('цены автомобилей с АКПП        :', price[auto][:5])
print('цены автомобилей с МКПП        :', price[~auto][:5])

# противоречивые записи: автоматическая коробка передач до 1970 г.
suspicious = (year < 1970) & auto
print('противоречивых записей:', suspicious.sum())
print(X[suspicious])   # маска применяется к строкам матрицы

годы выпуска до 1970 г.        : [1964. 1966. 1964. 1949. 1959. 1967. 1960. 1949. 1963. 1963.]
цены автомобилей с АКПП        : [ 7750. 17000. 11400. 11800.  5000.]
цены автомобилей с МКПП        : [8550. 2200. 6550. 4100. 3490.]
противоречивых записей: 8
[[   1953.  111111.    2000.    3500.]
 [   1900.       0.    1000.    1500.]
 [   1955.   25000.    2500.   10000.]
 [   1900. 1556666.    1212.   10000.]
 [   1900.    1877.    1400.    3000.]
 [   1900.       0.    1000.    1500.]
 [   1900.    1000.     100.   10000.]
 [   1954.  250000.     200.     750.]]


### 2.5. Пропуски

Отсутствующее значение обозначается `np.nan` (not a number). Проверка выполняется функцией `np.isnan`, которая возвращает логическую маску.

In [21]:
missing = np.isnan(engine)
print('пропусков в engine:', missing.sum())
print('записи с пропуском:')
print(X[missing][:3])

engine_known = engine[~missing]
print('значений без пропусков:', engine_known.size)

пропусков в engine: 205
записи с пропуском:
[[  2013. 172000.     nan   6299.]
 [  2009. 180000.     nan   5300.]
 [  2015. 169265.     nan  12650.]]
значений без пропусков: 40802


### Упражнение 2.2

1. Вычислите число автомобилей с автоматической коробкой передач: `n_auto`.
2. Выберите цены автомобилей с объёмом двигателя более 3000 см³: `price_big_engine`.
3. Вычислите число автомобилей, выпущенных до 1970 г.: `n_old`.

In [22]:
n_auto = auto.sum()
price_big_engine = price[engine > 3000]
n_old = (year < 1970).sum()

In [23]:
assert n_auto == 18590
assert price_big_engine.size == 1302
assert n_old == 89
print('Упражнение 2.2 выполнено')

Упражнение 2.2 выполнено


## 3. Базовые операции

### 3.1. Операции вектора с числом

Операция выполняется над каждым элементом вектора. Результат — вектор той же длины.

In [24]:
engine_l = engine / 1000          # объём двигателя в литрах
price_rub = price * RATE          # цена в рублях
print(engine_l[:5])
print(price_rub[:5])

[1.8 1.5 1.4 1.5 1.6]
[775000. 855000. 220000. 655000. 410000.]


### 3.2. Операции двух векторов

Операция выполняется над парами элементов с одинаковыми номерами. Векторы должны иметь одинаковую длину.

In [25]:
price_5 = price[:5]
engine_5 = engine[:5]
price_per_cm3 = price_5 / engine_5    # цена, приходящаяся на 1 см³ объёма двигателя
print(price_per_cm3)

try:
    price[:5] + price[:3]
except ValueError as err:
    print('векторы разной длины: ошибка ->', err)

[4.30555556 5.7        1.57142857 4.36666667 2.5625    ]
векторы разной длины: ошибка -> operands could not be broadcast together with shapes (5,) (3,) 


### 3.3. Линейная комбинация векторов

Каждый вектор умножается на число, результаты складываются. Эта операция используется в итоговом задании для прогноза цены.

In [26]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([10.0, 20.0, 30.0])
print(2 * a + 0.5 * b + 1)    # [2*1 + 0.5*10 + 1, 2*2 + 0.5*20 + 1, 2*3 + 0.5*30 + 1]
print(np.abs(a - b))          # модуль разности

[ 8. 15. 22.]
[ 9. 18. 27.]


### 3.4. Агрегирование

Функции агрегирования вычисляют одно число по всему вектору.

| Функция | Результат |
|---|---|
| `sum`, `mean`, `median` | сумма, среднее, медиана |
| `min`, `max` | минимум, максимум |
| `argmin`, `argmax` | номер минимального, максимального элемента |

При наличии пропусков `mean` возвращает `nan`. Для вычисления без учета пропусков используется `np.nanmean`.

In [27]:
print('средняя цена  :', price.mean())
print('медиана цены  :', np.median(price))
print('мин. / макс.  :', price.min(), price.max())

i_max = price.argmax()
print('самый дорогой автомобиль: запись', i_max, '->', X[i_max])

print('средний объём (mean)   :', engine.mean())
print('средний объём (nanmean):', np.nanmean(engine))

средняя цена  : 9727.109078937743


медиана цены  : 6600.0
мин. / макс.  : 1.0 10000000.0
самый дорогой автомобиль: запись 37125 -> [    2009.    57000.     1598. 10000000.]
средний объём (mean)   : nan
средний объём (nanmean): 1853.8636831527867


Для матрицы указывается ось агрегирования `axis`:
* `axis=0` — по записям, результат вычисляется для каждого признака;
* `axis=1` — по признакам, результат вычисляется для каждой записи.

Для матрицы `X` смысл имеет только `axis=0`: складывать год, пробег и цену одной записи бессмысленно.

In [28]:
print('средние по признакам:', np.nanmean(X, axis=0))
print('максимумы по признакам:', np.nanmax(X, axis=0))

средние по признакам: [  2007.9761748  456735.3023874    1853.86368315   9727.10907894]
максимумы по признакам: [2.021e+03 1.000e+08 9.999e+03 1.000e+07]


### Упражнение 3

1. Вычислите медиану пробега: `median_distance`.
2. Найдите год выпуска самого старого автомобиля: `oldest_year`.
3. Найдите цену самого старого автомобиля (используйте `argmin`): `oldest_price`.
4. **Вопрос.** Средняя цена значительно больше медианы. Какие значения в векторе `price` приводят к этому эффекту?

In [29]:
median_distance = np.median(distance)
oldest_year = year.min()
oldest_price = price[year.argmin()]

In [30]:
assert median_distance == np.median(distance)
assert oldest_year == 1900
assert oldest_price == price[year.argmin()]
print('Упражнение 3 выполнено')

Упражнение 3 выполнено


*Ответ на вопрос:*

## 4. Итоговое задание

Задание выполняется на полном наборе данных.

**Шаг 1.** Вычислите векторы признаков:
* `age` — возраст автомобиля, лет: `CURRENT_YEAR - year`;
* `km_year` — пробег в год, км: `distance / age`.

Вычислите средний возраст `mean_age` и средний пробег в год `mean_km_year`.

`CURRENT_YEAR` — год, следующий за годом последней записи набора данных (2021 + 1).

In [31]:
CURRENT_YEAR = 2022

In [32]:
age = CURRENT_YEAR - year
km_year = distance / age
mean_age = age.mean()
mean_km_year = km_year.mean()
print(f'средний возраст: {mean_age:.1f} лет, средний пробег в год: {mean_km_year:.0f} км')

средний возраст: 14.0 лет, средний пробег в год: 30098 км


In [33]:
assert age.shape == year.shape and age.min() == 1
assert km_year.shape == year.shape
assert abs(mean_age - (CURRENT_YEAR - year).mean()) < 1e-9
print('Шаг 1 выполнен')

Шаг 1 выполнен


**Шаг 2.** Постройте логическую маску `clean`, в которой значение `True` имеют записи без противоречий:
* объём двигателя известен (не `nan`);
* объём двигателя от 200 до 5000 см³ включительно;
* цена больше 100 и меньше 100 000 евро;
* не выполняется условие «пробег менее 1000 км и возраст более 1 года».

Примените маску к векторам `age`, `engine`, `price`, `auto`. Результаты сохраните в `age_c`, `engine_c`, `price_c`, `auto_c`.

In [34]:
clean = (~np.isnan(engine)
         & (engine >= 200) & (engine <= 5000)
         & (price > 100) & (price < 100_000)
         & ~((distance < 1000) & (age > 1)))

age_c = age[clean]
engine_c = engine[clean]
price_c = price[clean]
auto_c = auto[clean]
print('записей после очистки:', clean.sum(), 'из', clean.size)

записей после очистки: 36472 из 41007


In [35]:
assert clean.sum() == 36472
assert age_c.size == engine_c.size == price_c.size == auto_c.size == 36472
assert not np.isnan(engine_c).any()
print('Шаг 2 выполнен')

Шаг 2 выполнен


**Шаг 3.** Вычислите среднюю цену автомобилей с автоматической коробкой передач `mean_price_auto` и с механической `mean_price_manual`.

In [36]:
mean_price_auto = price_c[auto_c].mean()
mean_price_manual = price_c[~auto_c].mean()
print(f'АКПП: {mean_price_auto:.0f} евро, МКПП: {mean_price_manual:.0f} евро')

АКПП: 14631 евро, МКПП: 5191 евро


In [37]:
assert abs(mean_price_auto - price_c[auto_c].mean()) < 1e-6
assert abs(mean_price_manual - price_c[~auto_c].mean()) < 1e-6
print('Шаг 3 выполнен')

Шаг 3 выполнен


**Шаг 4.** Вычислите прогноз цены по формуле

$$\hat{p} = w_{age} \cdot age + w_{engine} \cdot engine + b,$$

где $\hat{p}$ — прогноз цены, евро; $age$ — возраст, лет; $engine$ — объём двигателя, см³; $w_{age}$, $w_{engine}$ — коэффициенты признаков; $b$ — смещение, евро.

Прогноз вычисляется для всех записей одновременно как линейная комбинация векторов (раздел 3.3). Результат сохраните в `price_pred`.

In [38]:
W_AGE = -500.0     # евро за 1 год возраста
W_ENGINE = 4.0     # евро за 1 см³ объёма двигателя
B = 5000.0         # евро

In [39]:
price_pred = W_AGE * age_c + W_ENGINE * engine_c + B
print(price_pred[:5])
print(price_c[:5])

[6700. 7000. 6000. 3400. 2800.]
[7750. 8550. 6550. 4100. 3490.]


In [40]:
assert price_pred.shape == price_c.shape
assert abs(price_pred[0] - (W_AGE * age_c[0] + W_ENGINE * engine_c[0] + B)) < 1e-9
print('Шаг 4 выполнен')

Шаг 4 выполнен


**Шаг 5.** Вычислите среднюю абсолютную ошибку прогноза

$$MAE = \frac{1}{n}\sum_{i=1}^{n} |\hat{p}_i - p_i|,$$

где $n$ — число записей; $p_i$ — фактическая цена $i$-й записи; $\hat{p}_i$ — прогноз цены $i$-й записи.

Для сравнения вычислите ошибку `mae_mean` прогноза, в котором для всех записей используется одно значение — средняя цена `price_c.mean()`.

In [41]:
mae = np.mean(np.abs(price_pred - price_c))
mae_mean = np.mean(np.abs(price_c.mean() - price_c))
print(f'MAE формулы       : {mae:.0f} евро')
print(f'MAE средней цены  : {mae_mean:.0f} евро')

MAE формулы       : 4588 евро
MAE средней цены  : 6303 евро


In [42]:
assert abs(mae - np.mean(np.abs(price_pred - price_c))) < 1e-6
assert mae < mae_mean
print('Шаг 5 выполнен')

Шаг 5 выполнен


**Шаг 6 (необязательный).** Измените значения `W_AGE`, `W_ENGINE`, `B` и повторно выполните шаги 4–5. Найдите значения коэффициентов, при которых `mae` меньше 4300 евро.

Подбор коэффициентов по данным называется **обучением модели**. Обучение модели регрессии рассматривается в последующих лабораторных работах курса.

## Контрольные вопросы

1. Чем вектор NumPy отличается от списка Python?
2. Что возвращает выражение `price > 10000`? Как с его помощью выбрать подмножество записей?
3. Что обозначают строка и столбец матрицы `X`?
4. Почему `engine.mean()` возвращает `nan`? Как вычислить среднее без учета пропусков?
5. Что показывает сравнение `mae` и `mae_mean`?